# Earnings Call EDA + FinBERT Sentiment Scoring

**Pipeline:** Structural EDA → LM dictionary baseline → FinBERT scoring → CEO/CFO divergence signal

**Assumes:** A `calls.csv` already exists with the transformed transcript data (one row per speaking turn).

**Required columns:** `componentorder`, `transcriptcomponenttypename`, `transcriptpersonname`, `title`, `speakertypeid`, `componenttext`, `ticker`, `companyname`, `mostimportantdateutc`, `quarter`, `close_to_open_return`

**Install deps if needed:**
```bash
pip install transformers torch pandas plotly scipy nltk
```

---
## 0 · Imports & Config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

# ── Dark theme config (MEI-inspired) ─────────────────────────────────────────
BG_PRIMARY   = '#0d0d14'
BG_SECONDARY = '#1a1a28'
ACCENT_GOLD  = '#f5a623'
ACCENT_TEAL  = '#4ecdc4'
ACCENT_RED   = '#f87171'
ACCENT_GREEN = '#34d399'
TEXT_PRIMARY = '#e8e8f0'
TEXT_MUTED   = '#8888aa'
GRID_COLOR   = '#1e1e30'
BORDER_COLOR = '#2a2a42'

PALETTE = [ACCENT_GOLD, ACCENT_TEAL, '#a78bfa', ACCENT_GREEN, '#f472b6', '#60a5fa', '#fb923c']

ROLE_COLORS  = {'CEO': ACCENT_GOLD, 'CFO': ACCENT_TEAL, 'Other': '#a78bfa'}

LAYOUT_BASE = dict(
    paper_bgcolor = BG_PRIMARY,
    plot_bgcolor  = BG_PRIMARY,
    font          = dict(family='sans-serif', color=TEXT_PRIMARY, size=12),
    title_font    = dict(color=TEXT_PRIMARY, size=15),
    legend        = dict(bgcolor=BG_SECONDARY, bordercolor=BORDER_COLOR,
                         borderwidth=1, font=dict(color=TEXT_MUTED, size=11)),
    xaxis = dict(gridcolor=GRID_COLOR, linecolor=BORDER_COLOR,
                 tickcolor=BORDER_COLOR, tickfont=dict(color=TEXT_MUTED)),
    yaxis = dict(gridcolor=GRID_COLOR, linecolor='rgba(0,0,0,0)',
                 tickfont=dict(color=TEXT_MUTED)),
    margin = dict(l=60, r=30, t=60, b=50),
)

def apply_theme(fig, title=None, xlab=None, ylab=None, height=400):
    """Apply dark theme to a plotly figure."""
    fig.update_layout(**LAYOUT_BASE, height=height)
    if title: fig.update_layout(title=dict(text=title, font=dict(color=TEXT_PRIMARY, size=15)))
    if xlab:  fig.update_xaxes(title_text=xlab, title_font=dict(color=TEXT_MUTED))
    if ylab:  fig.update_yaxes(title_text=ylab, title_font=dict(color=TEXT_MUTED))
    return fig

print('Imports ready.')

---
## 1 · Load & Transform Data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
DATA_PATH = 'calls.csv'   # adjust path as needed

calls = pd.read_csv(DATA_PATH)

# ── Normalise role from free-text title field ─────────────────────────────────
def normalise_role(title):
    if not isinstance(title, str): return 'Other'
    t = title.lower()
    if re.search(r'ceo|chief executive|president', t): return 'CEO'
    if re.search(r'cfo|chief financial|finance officer', t): return 'CFO'
    if re.search(r'coo|chief operating', t): return 'COO'
    if re.search(r'operator', t): return 'Operator'
    return 'Other'

calls['role'] = calls['title'].apply(normalise_role)

# ── Section label ─────────────────────────────────────────────────────────────
section_map = {
    'Presenter Speech': 'Prepared Remarks',
    'Answer':           'Q&A',
    'Question':         'Q&A',
}
calls['section'] = calls['transcriptcomponenttypename'].map(section_map).fillna(
    calls['transcriptcomponenttypename']
)

# ── Text metrics ──────────────────────────────────────────────────────────────
calls['word_count']   = calls['componenttext'].str.split().str.len()
calls['char_count']   = calls['componenttext'].str.len()
calls['sent_count']   = calls['componenttext'].apply(
    lambda x: len(sent_tokenize(str(x))) if isinstance(x, str) else 0
)
calls['avg_sent_len'] = calls['word_count'] / calls['sent_count'].clip(lower=1)

# ── Dates ─────────────────────────────────────────────────────────────────────
calls['call_date'] = pd.to_datetime(calls['mostimportantdateutc']).dt.date
calls['quarter']   = pd.to_datetime(calls['quarter']).dt.to_period('Q')

print(f'Loaded {len(calls):,} turns | {calls["transcriptid"].nunique()} call(s)')
print(f'Tickers: {calls["ticker"].unique()}')
print(f'Roles:   {calls["role"].value_counts().to_dict()}')
print(f'Sections:{calls["section"].value_counts().to_dict()}')

calls[['componentorder','transcriptpersonname','role','section',
       'word_count','sent_count','avg_sent_len']].head(8)

---
## 2 · Structural EDA

In [ ]:
# ── 2a. Summary stats by speaker ──────────────────────────────────────────────
speaker_stats = (
    calls
    .groupby(['transcriptpersonname', 'role', 'section'])
    .agg(
        n_turns      = ('componentorder', 'count'),
        total_words  = ('word_count', 'sum'),
        mean_words   = ('word_count', 'mean'),
        median_words = ('word_count', 'median'),
        mean_sents   = ('sent_count', 'mean'),
        mean_sent_len= ('avg_sent_len', 'mean'),
    )
    .round(1)
    .reset_index()
)

print('=== Speaker stats ===')
print(speaker_stats.to_string(index=False))

print('\n=== Call metadata ===')
meta = calls[['ticker','companyname','call_date','close_to_open_return']].iloc[0]
for k, v in meta.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 2b. Turn length boxplot ───────────────────────────────────────────────────
fig = go.Figure()

for person, grp in calls.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    fig.add_trace(go.Box(
        y           = grp['word_count'],
        name        = f'{person} ({role})',
        marker_color= color,
        boxpoints   = 'all',
        jitter      = 0.3,
        pointpos    = -1.6,
        line_color  = color,
        fillcolor   = color + '33',
    ))

apply_theme(fig,
    title = 'Turn Length Distribution by Speaker',
    xlab  = 'Speaker',
    ylab  = 'Words per Turn',
    height= 420,
)
fig.show()

In [ ]:
# ── 2c. Talk-time flow over component order ───────────────────────────────────
# Find Q&A boundary
qa_start = calls.loc[calls['section'] == 'Q&A', 'componentorder'].min()

fig = go.Figure()

for person, grp in calls.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    fig.add_trace(go.Bar(
        x     = grp['componentorder'],
        y     = grp['word_count'],
        name  = f'{person} ({role})',
        marker_color = color + 'cc',
        hovertemplate = '<b>%{text}</b><br>Order: %{x}<br>Words: %{y}<extra></extra>',
        text  = [person] * len(grp),
    ))

# Q&A boundary line
if not np.isnan(qa_start):
    fig.add_vline(
        x           = qa_start,
        line_dash   = 'dash',
        line_color  = TEXT_MUTED,
        line_width  = 1.5,
        annotation_text = 'Q&A starts',
        annotation_position = 'top right',
        annotation_font = dict(color=TEXT_MUTED, size=11),
    )

apply_theme(fig,
    title  = 'Talk-Time Flow by Component Order',
    xlab   = 'Component Order',
    ylab   = 'Word Count',
    height = 380,
)
fig.update_layout(barmode='stack')
fig.show()

---
## 3 · LM Dictionary Sentiment (Baseline)

Loughran-McDonald (2011) dictionary — purpose-built for financial text. Categories: `positive`, `negative`, `uncertainty`, `litigious`, `constraining`, `superfluous`.

In [ ]:
# ── 3a. Load LM dictionary ────────────────────────────────────────────────────
# Download from: https://sraf.nd.edu/loughranmcdonald-master-dictionary/
# Or install via: pip install pysentiment2
# Here we use a lightweight inline version for the key categories

# Option A — pysentiment2 (recommended, wraps official LM list)
try:
    import pysentiment2 as ps
    lm = ps.LM()
    USE_PYSENTIMENT = True
    print('Using pysentiment2 LM dictionary.')
except ImportError:
    USE_PYSENTIMENT = False
    print('pysentiment2 not found — pip install pysentiment2')
    print('Falling back to manual word lists (limited).')

# Option B — manual fallback word lists (extend as needed)
LM_POSITIVE = {
    'achieve','achievement','best','better','beneficial','bonus','confidence',
    'deliver','driven','effective','efficient','excellent','exceptional',
    'expand','growth','improve','improvement','increase','innovative','leader',
    'opportunity','outperform','positive','profit','record','robust','solid',
    'strong','succeed','success','superior','sustainable','value'
}
LM_NEGATIVE = {
    'adverse','burden','challenge','competitive','concern','constrain',
    'contraction','decline','decrease','default','deficit','delay','difficult',
    'disappointing','dispute','drop','fall','failed','headwind','impaired',
    'impairment','inadequate','insufficient','issue','loss','miss','negative',
    'penalty','pressure','problem','reduce','risk','shortfall','uncertain',
    'underperform','volatility','weak','weakness','write-off'
}
LM_UNCERTAINTY = {
    'approximately','appears','assume','assumption','believe','contingent',
    'depend','estimate','expect','forecast','guidance','hope','if','indicate',
    'likely','may','maybe','might','outlook','pending','plan','possible',
    'potentially','predict','project','seem','should','suppose','uncertain',
    'unclear','unknown','unless','volatile','whether'
}

def score_lm_turn(text):
    """Score a turn using LM dictionary. Returns dict of category counts + derived scores."""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return {'lm_pos':0,'lm_neg':0,'lm_unc':0,'lm_net':0,'lm_unc_pct':0,'lm_n_words':0}

    if USE_PYSENTIMENT:
        tokens = lm.tokenize(text)
        score  = lm.get_score(tokens)
        pos = score.get('Positive', 0)
        neg = score.get('Negative', 0)
        unc = score.get('Uncertainty', 0)
        n   = max(len(tokens), 1)
    else:
        words = re.findall(r'[a-z]+', text.lower())
        pos   = sum(1 for w in words if w in LM_POSITIVE)
        neg   = sum(1 for w in words if w in LM_NEGATIVE)
        unc   = sum(1 for w in words if w in LM_UNCERTAINTY)
        n     = max(len(words), 1)

    total_sent = pos + neg
    return {
        'lm_pos':     pos,
        'lm_neg':     neg,
        'lm_unc':     unc,
        'lm_n_words': n,
        'lm_net':     (pos - neg) / max(total_sent, 1),
        'lm_unc_pct': unc / n,
    }

lm_scores = calls['componenttext'].apply(score_lm_turn).apply(pd.Series)
calls = pd.concat([calls, lm_scores], axis=1)

print('LM scores computed.')
calls[['transcriptpersonname','role','section','lm_net','lm_unc_pct']].round(3)

In [ ]:
# ── 3b. LM net sentiment over component order ─────────────────────────────────
fig = go.Figure()

fig.add_hline(y=0, line_dash='dash', line_color=BORDER_COLOR, line_width=1)

for person, grp in calls.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    grp   = grp.sort_values('componentorder')
    fig.add_trace(go.Scatter(
        x    = grp['componentorder'],
        y    = grp['lm_net'],
        mode = 'lines+markers',
        name = f'{person} ({role})',
        line = dict(color=color, width=2),
        marker = dict(
            color  = color,
            size   = grp['word_count'].clip(upper=400) / 40 + 4,
            line   = dict(color=BG_PRIMARY, width=1.5),
        ),
        hovertemplate = (
            '<b>%{customdata[0]}</b><br>'
            'Order: %{x}<br>LM Net: %{y:.3f}<br>'
            'Section: %{customdata[1]}<br>Words: %{customdata[2]}<extra></extra>'
        ),
        customdata = grp[['transcriptpersonname','section','word_count']].values,
    ))

if not np.isnan(qa_start):
    fig.add_vline(x=qa_start, line_dash='dash', line_color=TEXT_MUTED,
                  line_width=1, annotation_text='Q&A',
                  annotation_font=dict(color=TEXT_MUTED, size=10))

apply_theme(fig,
    title  = 'LM Net Sentiment by Turn (Baseline)',
    xlab   = 'Component Order',
    ylab   = 'Net Sentiment  (pos − neg) / total',
    height = 420,
)
fig.update_layout(legend=dict(x=0.01, y=0.99))
fig.show()

In [ ]:
# ── 3c. LM category breakdown stacked bar ────────────────────────────────────
# Aggregate by speaker
lm_agg = (
    calls
    .groupby(['transcriptpersonname','role'])
    [['lm_pos','lm_neg','lm_unc']]
    .sum()
    .reset_index()
)
lm_agg['total'] = lm_agg[['lm_pos','lm_neg','lm_unc']].sum(axis=1)
lm_agg[['pct_pos','pct_neg','pct_unc']] = (
    lm_agg[['lm_pos','lm_neg','lm_unc']].div(lm_agg['total'], axis=0) * 100
)

speakers = lm_agg['transcriptpersonname'].tolist()

fig = go.Figure(data=[
    go.Bar(name='Positive',    x=speakers, y=lm_agg['pct_pos'], marker_color=ACCENT_GREEN),
    go.Bar(name='Negative',    x=speakers, y=lm_agg['pct_neg'], marker_color=ACCENT_RED),
    go.Bar(name='Uncertainty', x=speakers, y=lm_agg['pct_unc'], marker_color=ACCENT_GOLD),
])

apply_theme(fig,
    title  = 'LM Sentiment Category Mix by Speaker',
    xlab   = 'Speaker',
    ylab   = '% of Sentiment Words',
    height = 380,
)
fig.update_layout(barmode='group')
fig.show()

print('\n=== LM Summary by Speaker ===')
print(lm_agg[['transcriptpersonname','role','pct_pos','pct_neg','pct_unc']].round(2).to_string(index=False))

---
## 4 · FinBERT Sentiment Scoring

**Model:** `ProsusAI/finbert` — BERT fine-tuned on financial text. Outputs `positive / negative / neutral` probabilities per text chunk.

Long turns are split on sentence boundaries into ≤450-token chunks and scores are averaged across chunks to avoid truncation artifacts.

In [ ]:
# ── 4a. Load FinBERT ──────────────────────────────────────────────────────────
from transformers import pipeline, AutoTokenizer
import torch

MODEL_NAME = 'ProsusAI/finbert'

print(f'Loading {MODEL_NAME}... (~400MB on first run, cached after)')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
finbert   = pipeline(
    'text-classification',
    model     = MODEL_NAME,
    tokenizer = tokenizer,
    top_k     = None,    # return all 3 class probs
    device    = 0 if torch.cuda.is_available() else -1,
)
print(f'FinBERT loaded. CUDA: {torch.cuda.is_available()}')

def chunk_text(text, max_tokens=450):
    """Split text into sentence-level chunks under max_tokens."""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return []
    sentences = sent_tokenize(text)
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        n = len(tokenizer.tokenize(sent))
        if current_len + n > max_tokens and current:
            chunks.append(' '.join(current))
            current, current_len = [sent], n
        else:
            current.append(sent)
            current_len += n
    if current:
        chunks.append(' '.join(current))
    return chunks if chunks else [text[:1500]]

def score_turn_finbert(text):
    """Score a speaking turn. Returns dict with mean pos/neg/neu probabilities."""
    chunks = chunk_text(text)
    if not chunks:
        return {'fb_positive': None, 'fb_negative': None, 'fb_neutral': None}

    results = finbert(chunks)
    pos = neg = neu = 0.0
    for chunk_result in results:
        scores = {r['label'].lower(): r['score'] for r in chunk_result}
        pos   += scores.get('positive', 0)
        neg   += scores.get('negative', 0)
        neu   += scores.get('neutral',  0)
    n = len(results)
    return {
        'fb_positive': round(pos / n, 4),
        'fb_negative': round(neg / n, 4),
        'fb_neutral':  round(neu  / n, 4),
    }

print(f'Scoring {len(calls)} turns...')
fb_scores = calls['componenttext'].apply(score_turn_finbert).apply(pd.Series)
calls     = pd.concat([calls, fb_scores], axis=1)

# Derived columns
calls['fb_net']   = calls['fb_positive'] - calls['fb_negative']
calls['fb_label'] = calls[['fb_positive','fb_negative','fb_neutral']].idxmax(axis=1) \
                         .str.replace('fb_', '')

print('Done.')
calls[['componentorder','role','section','fb_positive','fb_negative','fb_neutral','fb_label']].round(3)

In [ ]:
# ── 4b. FinBERT summary stats ─────────────────────────────────────────────────
fb_speaker = (
    calls
    .groupby(['transcriptpersonname','role'])
    .agg(
        n_turns      = ('fb_net', 'count'),
        mean_pos     = ('fb_positive', 'mean'),
        mean_neg     = ('fb_negative', 'mean'),
        mean_neu     = ('fb_neutral',  'mean'),
        mean_net     = ('fb_net', 'mean'),
        pct_positive = ('fb_label', lambda x: (x == 'positive').mean()),
        pct_negative = ('fb_label', lambda x: (x == 'negative').mean()),
    )
    .round(3)
    .reset_index()
)

fb_section = (
    calls
    .groupby('section')
    .agg(
        n_turns  = ('fb_net', 'count'),
        mean_pos = ('fb_positive', 'mean'),
        mean_neg = ('fb_negative', 'mean'),
        mean_net = ('fb_net', 'mean'),
    )
    .round(3)
    .reset_index()
)

print('=== FinBERT by Speaker ===')
print(fb_speaker.to_string(index=False))
print('\n=== FinBERT by Section ===')
print(fb_section.to_string(index=False))

---
## 5 · FinBERT Visualisations

In [ ]:
# ── 5a. Probability stack per turn (faceted by speaker) ───────────────────────
roles_present = calls['role'].unique()
n_roles = len([r for r in ['CEO','CFO'] if r in roles_present])

fig = make_subplots(
    rows=1, cols=n_roles,
    subplot_titles=[r for r in ['CEO','CFO'] if r in roles_present],
    shared_yaxes=True,
)

for col_idx, role in enumerate([r for r in ['CEO','CFO'] if r in roles_present], 1):
    grp = calls[calls['role'] == role].sort_values('componentorder')
    labels = [f'T{o} ({s[:3]})' for o, s in zip(grp['componentorder'], grp['section'])]

    for label, color, col_key in [
        ('Positive', ACCENT_GREEN,  'fb_positive'),
        ('Neutral',  TEXT_MUTED,    'fb_neutral'),
        ('Negative', ACCENT_RED,    'fb_negative'),
    ]:
        fig.add_trace(
            go.Bar(
                name       = label,
                x          = labels,
                y          = grp[col_key].round(3),
                marker_color = color,
                showlegend   = (col_idx == 1),
                hovertemplate = f'<b>{label}</b>: %{{y:.3f}}<extra></extra>',
            ),
            row=1, col=col_idx,
        )

apply_theme(fig, title='FinBERT Probability Stack by Turn', height=460)
fig.update_layout(
    barmode='stack',
    xaxis = dict(tickangle=-40, tickfont=dict(size=9, color=TEXT_MUTED)),
    xaxis2= dict(tickangle=-40, tickfont=dict(size=9, color=TEXT_MUTED)),
)
fig.update_annotations(font=dict(color=TEXT_PRIMARY, size=12))
fig.show()

In [ ]:
# ── 5b. FinBERT net score over component order ────────────────────────────────
fig = go.Figure()
fig.add_hline(y=0, line_dash='dash', line_color=BORDER_COLOR, line_width=1)

for person, grp in calls.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    grp   = grp.sort_values('componentorder').dropna(subset=['fb_net'])

    fig.add_trace(go.Scatter(
        x    = grp['componentorder'],
        y    = grp['fb_net'].round(3),
        mode = 'lines+markers',
        name = f'{person} ({role})',
        line = dict(color=color, width=2.5),
        marker = dict(
            color  = color,
            size   = grp['word_count'].clip(upper=400) / 40 + 5,
            line   = dict(color=BG_PRIMARY, width=1.5),
        ),
        hovertemplate = (
            '<b>%{customdata[0]}</b>  T%{x}<br>'
            'FinBERT net: %{y:.3f}<br>'
            'Label: %{customdata[1]}<br>'
            'Section: %{customdata[2]}<extra></extra>'
        ),
        customdata = grp[['transcriptpersonname','fb_label','section']].values,
    ))

if not np.isnan(qa_start):
    fig.add_vline(x=qa_start, line_dash='dash', line_color=TEXT_MUTED, line_width=1,
                  annotation_text='Q&A', annotation_font=dict(color=TEXT_MUTED, size=10))

apply_theme(fig,
    title  = 'FinBERT Net Sentiment Over Call  (pos − neg)',
    xlab   = 'Component Order',
    ylab   = 'Net Score  (positive − negative)',
    height = 440,
)
fig.show()

In [ ]:
# ── 5c. LM vs FinBERT cross-validation scatter ────────────────────────────────
compare = calls.dropna(subset=['fb_net','lm_net']).copy()
corr, pval = stats.pearsonr(compare['lm_net'], compare['fb_net'])

fig = go.Figure()

for person, grp in compare.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    fig.add_trace(go.Scatter(
        x    = grp['lm_net'].round(3),
        y    = grp['fb_net'].round(3),
        mode = 'markers',
        name = f'{person} ({role})',
        marker = dict(
            color  = color,
            size   = grp['word_count'].clip(upper=400) / 40 + 6,
            opacity= 0.85,
            line   = dict(color=BG_PRIMARY, width=1),
        ),
        hovertemplate = (
            '<b>%{customdata[0]}</b><br>'
            'LM net: %{x:.3f}<br>FinBERT net: %{y:.3f}<br>'
            'Words: %{customdata[1]}<extra></extra>'
        ),
        customdata = grp[['transcriptpersonname','word_count']].values,
    ))

# Diagonal reference line
xy_min = min(compare['lm_net'].min(), compare['fb_net'].min()) - 0.05
xy_max = max(compare['lm_net'].max(), compare['fb_net'].max()) + 0.05
fig.add_trace(go.Scatter(
    x=[xy_min, xy_max], y=[xy_min, xy_max],
    mode='lines', line=dict(color=BORDER_COLOR, dash='dash', width=1),
    showlegend=False, hoverinfo='skip',
))

apply_theme(fig,
    title  = f'LM vs FinBERT Net Sentiment  (r = {corr:.3f}, p = {pval:.3f})',
    xlab   = 'LM Net Sentiment',
    ylab   = 'FinBERT Net Score',
    height = 460,
)
fig.add_hline(y=0, line_dash='dot', line_color=BORDER_COLOR, line_width=0.8)
fig.add_vline(x=0, line_dash='dot', line_color=BORDER_COLOR, line_width=0.8)
fig.show()

print(f'Pearson r (LM net vs FinBERT net): {corr:.3f}  (p={pval:.3f})')
print('Disagreement cases (different sign):')
print(compare.loc[
    np.sign(compare['lm_net']) != np.sign(compare['fb_net']),
    ['transcriptpersonname','componentorder','section','lm_net','fb_net','fb_label','word_count']
].round(3).to_string(index=False))

In [ ]:
# ── 5d. Uncertainty vs FinBERT P(negative) ────────────────────────────────────
fig = go.Figure()

for person, grp in compare.groupby('transcriptpersonname'):
    role  = grp['role'].iloc[0]
    color = ROLE_COLORS.get(role, '#a78bfa')
    fig.add_trace(go.Scatter(
        x    = grp['lm_unc_pct'].round(4),
        y    = grp['fb_negative'].round(3),
        mode = 'markers',
        name = f'{person} ({role})',
        marker = dict(
            color   = color,
            size    = 10,
            opacity = 0.85,
            line    = dict(color=BG_PRIMARY, width=1),
        ),
        hovertemplate = (
            '<b>%{customdata}</b><br>'
            'LM uncertainty: %{x:.3f}<br>'
            'FinBERT P(neg): %{y:.3f}<extra></extra>'
        ),
        customdata = grp['transcriptpersonname'],
    ))

apply_theme(fig,
    title  = 'LM Uncertainty vs FinBERT P(negative)',
    xlab   = 'LM Uncertainty  (% of total words)',
    ylab   = 'FinBERT P(negative)',
    height = 420,
)
fig.show()

---
## 6 · CEO / CFO Divergence Signal

Within-call preview of **Idea #2**. In the multi-call corpus you'd z-score each executive against their personal baseline; here we compute raw deltas as a starting point.

In [ ]:
# ── 6a. Per-role aggregates ────────────────────────────────────────────────────
role_agg = (
    calls
    .groupby('role')
    .agg(
        n_turns      = ('fb_net', 'count'),
        mean_fb_net  = ('fb_net', 'mean'),
        mean_fb_pos  = ('fb_positive', 'mean'),
        mean_fb_neg  = ('fb_negative', 'mean'),
        mean_lm_net  = ('lm_net', 'mean'),
        mean_lm_unc  = ('lm_unc_pct', 'mean'),
        total_words  = ('word_count', 'sum'),
    )
    .round(4)
)

print('=== Role-Level Aggregates ===')
print(role_agg.to_string())

# ── 6b. Divergence metrics ────────────────────────────────────────────────────
if 'CEO' in role_agg.index and 'CFO' in role_agg.index:
    ceo = role_agg.loc['CEO']
    cfo = role_agg.loc['CFO']

    divergence = {
        'FinBERT net delta (CEO - CFO)':      round(ceo['mean_fb_net'] - cfo['mean_fb_net'], 4),
        'FinBERT P(neg) delta (CEO - CFO)':   round(ceo['mean_fb_neg'] - cfo['mean_fb_neg'], 4),
        'LM net delta (CEO - CFO)':           round(ceo['mean_lm_net'] - cfo['mean_lm_net'], 4),
        'LM uncertainty delta (CEO - CFO)':   round(ceo['mean_lm_unc'] - cfo['mean_lm_unc'], 4),
        'Close-to-open return':               round(calls['close_to_open_return'].iloc[0], 5),
    }

    print('\n=== CEO / CFO Divergence ===')
    for k, v in divergence.items():
        print(f'  {k:<45} {v:+.4f}')
else:
    print('CEO or CFO role not found — check role normalisation.')

In [ ]:
# ── 6c. Radar chart — CEO vs CFO sentiment profile ────────────────────────────
if 'CEO' in role_agg.index and 'CFO' in role_agg.index:
    categories   = ['FinBERT Positive','FinBERT Negative','FinBERT Neutral',
                     'LM Net (scaled)','LM Uncertainty']

    # Scale LM net to [0,1] range for radar readability
    lm_net_min = role_agg['mean_lm_net'].min()
    lm_net_max = role_agg['mean_lm_net'].max()
    lm_net_range = max(lm_net_max - lm_net_min, 0.01)

    def radar_vals(row):
        return [
            row['mean_fb_pos'],
            row['mean_fb_neg'],
            row['mean_fb_neg'],   # neutral
            (row['mean_lm_net'] - lm_net_min) / lm_net_range,
            row['mean_lm_unc'],
        ]

    fig = go.Figure()
    for role, color in [('CEO', ACCENT_GOLD), ('CFO', ACCENT_TEAL)]:
        vals = radar_vals(role_agg.loc[role])
        fig.add_trace(go.Scatterpolar(
            r    = vals + [vals[0]],
            theta= categories + [categories[0]],
            name = role,
            line = dict(color=color, width=2),
            fill = 'toself',
            fillcolor = color + '22',
        ))

    fig.update_layout(
        polar = dict(
            bgcolor     = BG_SECONDARY,
            radialaxis  = dict(visible=True, range=[0,1],
                               gridcolor=BORDER_COLOR, tickfont=dict(color=TEXT_MUTED, size=9)),
            angularaxis = dict(gridcolor=BORDER_COLOR, tickfont=dict(color=TEXT_PRIMARY, size=11)),
        ),
        paper_bgcolor = BG_PRIMARY,
        font          = dict(color=TEXT_PRIMARY),
        title         = dict(text='CEO vs CFO — Sentiment Profile', font=dict(color=TEXT_PRIMARY, size=14)),
        legend        = dict(bgcolor=BG_SECONDARY, bordercolor=BORDER_COLOR, borderwidth=1,
                             font=dict(color=TEXT_MUTED)),
        height        = 460,
    )
    fig.show()

---
## 7 · Save Scored Data

In [ ]:
# ── Save full scored dataframe ─────────────────────────────────────────────────
# Drop raw text for the lightweight version, keep it for the full version

OUTPUT_PATH = 'calls_finbert_scored.csv'

cols_to_save = [
    'transcriptid','transcriptcomponentid','componentorder',
    'transcriptpersonname','role','section','title',
    'ticker','companyname','call_date','quarter',
    'word_count','sent_count','avg_sent_len',
    'lm_pos','lm_neg','lm_unc','lm_net','lm_unc_pct',
    'fb_positive','fb_negative','fb_neutral','fb_net','fb_label',
    'close_to_open_return',
]

out = calls[[c for c in cols_to_save if c in calls.columns]]
out.to_csv(OUTPUT_PATH, index=False)

print(f'Saved {len(out):,} rows to {OUTPUT_PATH}')
print(out[['role','fb_net','lm_net','fb_label']].groupby('role').mean(numeric_only=True).round(3))